# Indic Shield AI — Context-Aware Toxicity Classifier (Telugu / Tanglish)

This notebook is a **front-end to the code in `src/`**; it contains no model code of its own, so it always
matches what the app and the training scripts run.

**Problem.** Whether a comment is toxic often depends on the conversation: *"Champesav po!"* is praise after a
gaming clutch and incitement after *"that journalist should be beaten"*. The model therefore classifies a
**target** comment *given its preceding turns*, for Telugu, Romanized Telugu and code-mixed text.

| Section | What it does |
|---|---|
| 1 | Setup & data status |
| 2 | Preprocessing and model input assembly |
| 3 | Training (commands — run in a terminal) |
| 4 | Evaluation results (measured, from `reports/`) |
| 5 | Baselines & context ablation |
| 6 | Live moderation + attention heatmap |

## 1. Setup & data status

In [ ]:
import sys, json
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

import pandas as pd
from src.config import PROCESSED_DIR, CHECKPOINT_DIR, REPORTS_DIR, PRIMARY_MODEL, TOXICITY_LABELS, indic_supervised_sources
print("Backbone:", PRIMARY_MODEL)

rows = []
for name in ["stage1_train", "stage1_val", "dravidian_codemix", "cultural_safety_telugu", "indic_corp_telugu_mlm"]:
    p = PROCESSED_DIR / f"{name}.parquet"
    if p.exists():
        d = pd.read_parquet(p)
        rows.append({"dataset": name, "rows": len(d), **{l: f"{(d[l] == 1).mean():.1%}" for l in TOXICITY_LABELS if l in d}})
    else:
        rows.append({"dataset": name, "rows": "missing"})
for p in indic_supervised_sources():
    if p.stem.startswith("custom_"):
        d = pd.read_parquet(p); rows.append({"dataset": p.stem, "rows": len(d), "toxic": f"{(d['toxic'] == 1).mean():.1%}"})
pd.DataFrame(rows)

Missing data? Run `python run_pipeline.py --download-only` (Jigsaw falls back to the public `google/civil_comments`).
Your own Telugu data goes in `data/raw/custom/` (columns: `text`, `label`, optional `context`, `topic`, `split`).
Label values of `-1` mean *not annotated by that dataset*; they are masked out of the loss.

## 2. Preprocessing and model input assembly

In [ ]:
from src.preprocessing import IndicPreprocessor
pre = IndicPreprocessor(enable_transliteration=False)
for t in ["m@drch0d nuvvu chalaaaaa", "https://x.com @troll l@nja", "gooood movie bro"]:
    print(f"{t!r:35} -> {pre.clean_text(t)!r}")

In [ ]:
from src.model import encode_example, load_tokenizer
tok = load_tokenizer(PRIMARY_MODEL)
enc = encode_example(tok, "Champesav po!", history=["Aaa journalist bayataki vasthe kottali", "Vaadini thagalettali"], topic="Politics")
tokens = tok.convert_ids_to_tokens(enc["input_ids"])
print(" ".join(f"[{t}]" if m else t for t, m in zip(tokens, enc["target_mask"])))
print("\n[brackets] = target span used by target pooling / cross-attention. Context is truncated oldest-first; the target never is.")

## 3. Training (run in a terminal)

```bash
python run_pipeline.py --download-only          # data
python run_pipeline.py --stage1                  # English toxicity (toxic-chat + Jigsaw/Civil Comments)
python run_pipeline.py --stage2                  # Telugu MLM on IndicCorpV2
python run_pipeline.py --stage3                  # Dravidian code-mix + Telugu safety (+ your custom data)
python run_pipeline.py --stage4                  # contrastive thread pairs
python run_pipeline.py --evaluate --baselines
```
Each checkpoint's `metadata.json` stores its data, tuned thresholds and validation metrics:

In [ ]:
from src.model import read_checkpoint_metadata
summary = []
for name in ["stage1_best", "stage2_best", "stage3_best", "stage4_best"]:
    m = read_checkpoint_metadata(CHECKPOINT_DIR / name)
    vm = m.get("val_metrics", {})
    summary.append({"checkpoint": name, "trained": bool(m), "val macro-F1": vm.get("macro_f1@threshold"),
                    "held-out flip": vm.get("flip_accuracy"), "MLM ppl": m.get("perplexity"), "thresholds": m.get("thresholds")})
pd.DataFrame(summary)

## 4. Evaluation results (measured)

In [ ]:
res_path = REPORTS_DIR / "eval_results.json"
if not res_path.exists():
    print("No results yet — run: python run_pipeline.py --evaluate")
else:
    res = json.loads(res_path.read_text(encoding="utf-8"))
    print("Evaluated:", res["evaluated_at"], "| checkpoint:", res["checkpoint"]["checkpoint"])
    display(pd.DataFrame(res["slices"]).T)
    if "heldout_test" in res:
        ho = res["heldout_test"]
        print(f"Held-out test n={ho['n']:,}  macro-F1={ho['macro_f1@threshold']:.3f}")
        display(pd.DataFrame(ho["per_label"]).T)

Slices 1–4 are small hand-written sanity checks (see their `n`). The held-out test split is the headline number.

## 5. Baselines & context ablation

In [ ]:
bl_path = REPORTS_DIR / "baselines.json"
if not bl_path.exists():
    print("Run: python run_pipeline.py --baselines")
else:
    bl = json.loads(bl_path.read_text(encoding="utf-8"))
    print(f"Held-out test n={bl['n_test']:,}, toxic rate {bl['test_toxic_rate']:.1%}")
    display(pd.DataFrame(bl["methods"]).T[["f1", "precision", "recall", "roc_auc"]])
    if "context_ablation" in bl:
        ab = bl["context_ablation"]
        print(f"Contrastive flip with context: {ab['contrastive_flip_with_context']:.0%} (without context it is 0% by construction)")
        print(f"Mean P(toxic) shift, toxic vs safe parent: {ab['mean_prob_shift_toxic_minus_safe_parent']:+.3f}")
        display(pd.DataFrame(ab["rows"]))

## 6. Live moderation + attention heatmap

In [ ]:
from src.inference_pipeline import TwoStageModerationPipeline
pipe = TwoStageModerationPipeline()
print(pipe.info())

In [ ]:
import matplotlib.pyplot as plt

def show(target, history=(), topic=None):
    r = pipe.moderate(target, history_override=list(history), topic=topic)
    print(f"{r['action']}  score={r['toxicity_score']}  [{r['stage_routed']}]  {r['latency_ms']} ms\n{r['explanation']}")
    a = r.get("attention_analysis")
    if not a:
        return r
    toks = a["tokens"]
    plt.figure(figsize=(min(16, 0.45 * len(toks) + 2), 2.2))
    plt.bar(range(len(toks)), [t["attention"] for t in toks], color=["#0ea5e9" if t["is_target"] else "#a855f7" for t in toks])
    plt.xticks(range(len(toks)), [t["token"] for t in toks], rotation=70, fontsize=8)
    plt.ylabel("pooling weight"); plt.title("blue = target, purple = context"); plt.tight_layout(); plt.show()
    return r

if pipe.ready:
    show("Champesav po!", ["Ninna match lo 1v4 clutch chusava bro?"], "Gaming")
    show("Champesav po!", ["Aaa journalist bayataki vasthe kottali"], "Politics")